# End to end (repro)

A number in a manuscript names a claim. The claim names a run. The run names its outputs,
hashed when they were recorded.

Every cell runs the **published package**. `pip install reproducible-science` gets you the
same code.

## Install

In [ ]:
import piplite

await piplite.install(["results-cli==0.2.0", "reproducible-science==0.2.0"])
print("installed")

## The shell stand-in

`results seal x.json` becomes `cli('results.cli', 'seal', 'x.json')`.

In [ ]:
import sys


def cli(module, *args):
    old = sys.argv
    sys.argv = [module.split(".")[0], *args]
    try:
        __import__(module, fromlist=["main"]).main()
    except SystemExit:
        pass
    finally:
        sys.argv = old


print("ready")

## A result to make claims about

In [ ]:
import json
import os
import pathlib

os.makedirs("/tmp/demo", exist_ok=True)
os.chdir("/tmp/demo")
pathlib.Path("results.json").write_text(
    json.dumps({"primary": {"effect": 0.404, "n": 200}}, indent=2)
)
print(pathlib.Path("results.json").read_text())

## Seal the input, record the run, bind the claim

In [ ]:
cli("results.cli", "init")
cli("results.cli", "seal", "results.json", "--role", "input")
cli("results.cli", "run", "results.json", "--run-id", "exp_001")
cli("results.cli", "claim", "The primary effect was 0.404", "--run-id", "exp_001")

## The ledger is a hash chain with a length anchor

In [ ]:
cli("results.cli", "verify")

## Declare what the manuscript says, and check it

In [ ]:
pathlib.Path("repro.yaml").write_text("""schema_version: repro/1
project: demo
artifacts:
  - id: results
    path: results.json
claims:
  - id: primary
    text: The primary effect was 0.404
    evidence:
      - kind: metric
        artifact: results
        name: effect
        pointer: /primary/effect
        reported: "0.404"
""")
cli("repro.cli", "verify", "repro.yaml")

---
# Break it

Each cell breaks one link. What matters is *which kind of no* the tool says.

## The manuscript prints a different number

Read, and disagreed. That is a `mismatch` — a fact about the paper.

In [ ]:
text = pathlib.Path("repro.yaml").read_text()
pathlib.Path("repro.yaml").write_text(text.replace('"0.404"', '"0.410"'))
cli("repro.cli", "verify", "repro.yaml")
pathlib.Path("repro.yaml").write_text(text)

## The address resolves to nothing

Not a mismatch. Nothing was compared, so nothing disagreed.

In [ ]:
text = pathlib.Path("repro.yaml").read_text()
pathlib.Path("repro.yaml").write_text(text.replace("/primary/effect", "/primary/effect_size"))
cli("repro.cli", "verify", "repro.yaml")
pathlib.Path("repro.yaml").write_text(text)

## Delete a ledger entry, then try to launder it

Truncation forges no line, so the chain stays perfect and only the anchor disagrees.
`reanchor` refuses rather than recording the damage as authoritative.

In [ ]:
lines = pathlib.Path(".results/ledger.jsonl").read_text().splitlines()
pathlib.Path(".results/ledger.jsonl").write_text("\n".join(lines[:-1]) + "\n")
cli("results.cli", "verify")
print("--- trying to re-anchor over it ---")
cli("results.cli", "reanchor")

---
## Not possible in WebAssembly

`prereg freeze` records the commit a plan was frozen at. Git does not exist in
WebAssembly, so the command cannot run here.

```bash
pip install reproducible-science
```